This is a code for batch processing. We load the model from web-service-mlflow notebook and then apply it on the values to see how good-bad predictions are from the real values. 

In [50]:
import uuid
import pandas as pd

import mlflow

In [51]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
RUN_ID = '9c986ad616164252bd48902dce588b49'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [52]:
year = 2021
month = 1
taxi_type = 'green'


input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'
output_file = f'output/{taxi_type}-{year:04d}-{month:02d}.parquet'

#RUN_ID = os.getenv('RUN_ID', 'e1efc53e9bd149078b0c12aeaa6365df')

In [54]:
def generate_uuids(n):
    ride_ids = []
    for i in range(n):
        ride_ids.append(str(uuid.uuid4()))
    return ride_ids

def read_dataframe(filename: str):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    df['ride_id'] = generate_uuids(len(df))

    return df


def prepare_dictionaries(df: pd.DataFrame):
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [58]:
def load_model(run_id):
#    logged_model = f's3://mlflow-models-alexey/1/{RUN_ID}/artifacts/model'
    logged_model = f'runs:/{RUN_ID}/model'

    model = mlflow.pyfunc.load_model(logged_model)
    return model


#we don't train the model, we apply it. so no x_train and etc

def apply_model(input_file, run_id, output_file):

    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)

    
    model = load_model(run_id)
    y_pred = model.predict(dicts) # applying our model to dictionaries

    df_result = pd.DataFrame()
    df_result['ride_id'] = df['ride_id']
    df_result['lpep_pickup_datetime'] = df['lpep_pickup_datetime']
    df_result['PULocationID'] = df['PULocationID']
    df_result['DOLocationID'] = df['DOLocationID']
    df_result['actual_duration'] = df['duration']
    df_result['predicted_duration'] = y_pred
    df_result['diff'] = df_result['actual_duration'] - df_result['predicted_duration']
    df_result['model_version'] = run_id
    
    df_result.to_parquet(output_file, index=False)

#    return df_result

In [59]:
apply_model(input_file=input_file, run_id=RUN_ID, output_file=output_file)

In [57]:
!ls output/green-*

output/green-2021-01.parquet output/green-2021-03.parquet
output/green-2021-02.parquet


In [ ]:
#jupyter nbconvert --to script score.ipynb